# Análisis de filtrado de full text
Compara qué papers pasan o no el `FullTextFilter` (Step 6).

In [57]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.paper import Paper
from utils.intermediate_io import STEP4_FILE, STEP5_FILE, STEP6_FILE, load_model_list

In [58]:
papers_step4 = load_model_list(STEP4_FILE, Paper)
papers_step5 = load_model_list(STEP5_FILE, Paper)
papers_step6 = load_model_list(STEP6_FILE, Paper)

print(f"Step4 (raw full text): {len(papers_step4)}")
print(f"Step5 (clean text): {len(papers_step5)}")
print(f"Step6 (passed filter): {len(papers_step6)}")

Step4 (raw full text): 94
Step5 (clean text): 93
Step6 (passed filter): 81


In [59]:
def pid(p: Paper) -> str:
    return (p.doi or p.title).strip().lower()

def to_df(papers, step_name: str):
    return pd.DataFrame([{
        "paper_id": pid(p),
        "title": p.title,
        "doi": p.doi,
        "format": p.full_text.format.value if p.full_text else None,
        "chars": len(p.full_text.content) if (p.full_text and p.full_text.content) else 0,
        "source": p.source,
        "ft_retrieved_by": p.ft_retrieved_by,
        "step": step_name,
    } for p in papers])

df4 = to_df(papers_step4, "step4")
df5 = to_df(papers_step5, "step5")
df6 = to_df(papers_step6, "step6")

transitions = [
    ("step4", df4, "step5", df5),
    ("step5", df5, "step6", df6),
]

dropped_parts = []
for from_step, from_df, to_step, to_df_ in transitions:
    to_ids = set(to_df_["paper_id"])
    dropped = from_df[~from_df["paper_id"].isin(to_ids)].copy()
    dropped["from_step"] = from_step
    dropped["to_step"] = to_step
    dropped_parts.append(dropped)

dropped_all = pd.concat(dropped_parts, ignore_index=True)
dropped_all.head()

,paper_id,title,doi,format,chars,source,ft_retrieved_by,step,from_step,to_step
0,10.2174/9781608052608113020004,Structural Analysis of Fungal Glucans,10.2174/9781608052608113020004,pdf,652900,crossref,semantic_scholar,step4,step4,step5
1,10.2210/pdb4gdn/pdb,Structure of FmtA-like protein,10.2210/pdb4gdn/pdb,html,1826,crossref,openalex,step5,step5,step6
2,10.4324/9780080518060-8,Structural requirements,10.4324/9780080518060-8,html,419,crossref,openalex,step5,step5,step6
3,10.4324/9780080518060-9,Structural materials,10.4324/9780080518060-9,html,419,crossref,openalex,step5,step5,step6
4,10.1128/aac.43.9.2121,"Characterization of fmtA , a Gene That Modulat...",10.1128/aac.43.9.2121,xml,2594,openalex,pmc,step5,step5,step6


In [60]:
summary = (
    dropped_all.groupby(["from_step", "to_step", "format"]).agg(
        n=("paper_id", "count"),
        median_chars=("chars", "median"),
    )
    .reset_index()
    .sort_values(["from_step", "n"], ascending=[True, False])
)
summary

,from_step,to_step,format,n,median_chars
0,step4,step5,pdf,1,652900.0
1,step5,step6,html,9,818.0
3,step5,step6,xml,2,2447.5
2,step5,step6,pdf,1,577.0


In [61]:
cols = ["from_step", "to_step", "paper_id", "title", "doi", "format", "chars", "source", "ft_retrieved_by"]
dropped_all[cols].sort_values(["from_step", "to_step", "chars"]).head(50)


,from_step,to_step,paper_id,title,doi,format,chars,source,ft_retrieved_by
0,step4,step5,10.2174/9781608052608113020004,Structural Analysis of Fungal Glucans,10.2174/9781608052608113020004,pdf,652900,crossref,semantic_scholar
2,step5,step6,10.4324/9780080518060-8,Structural requirements,10.4324/9780080518060-8,html,419,crossref,openalex
3,step5,step6,10.4324/9780080518060-9,Structural materials,10.4324/9780080518060-9,html,419,crossref,openalex
11,step5,step6,10.1101/2023.01.08.522890,VICTORY for Cryo-EM: Ultra-High Atomic Resolut...,10.1101/2023.01.08.522890,pdf,577,crossref,openalex
7,step5,step6,10.1016/s1876-1623(25)00079-3,Title Page,10.1016/s1876-1623(25)00079-3,html,813,crossref,openalex
8,step5,step6,10.1016/s1876-1623(25)00078-1,Half Title Page,10.1016/s1876-1623(25)00078-1,html,818,crossref,openalex
9,step5,step6,10.55277/researchhub.97wgn08x.1,title title title title title,10.55277/researchhub.97wgn08x.1,html,818,crossref,openalex
12,step5,step6,10.1016/b978-1-4832-2843-3.50026-1,Conformation in Chitin–Protein Complexes,10.1016/b978-1-4832-2843-3.50026-1,html,846,crossref,openalex
5,step5,step6,10.1016/0022-2836(75)90212-0,Molecular structure determination by electron ...,10.1016/0022-2836(75)90212-0,html,893,elsevier,openalex
6,step5,step6,10.2210/pdb6bg9/pdb,HYBRID NMR/CRYO-EM STRUCTURE OF THE HIV-1 RNA ...,10.2210/pdb6bg9/pdb,html,1811,crossref,openalex


## Notas
- Compara pérdidas entre pasos consecutivos: `step4 -> step5` y `step5 -> step6`.
- `dropped_all` contiene el detalle de papers que NO pasan entre un paso y otro.
- La salida impresa agrupa por transición para facilitar revisión rápida en terminal/notebook.